In [ ]:
%pip install keras-tuner

In [ ]:
%pip install tensorflow

In [ ]:
%pip install scikit-learn

In [ ]:
%pip install pandas

In [ ]:
%pip install tensorboard

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
import keras_tuner as kt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
data = pd.read_csv('../Datasets/huge_1M_titanic.csv')

In [ ]:
data = data.sample(10000, random_state=42)

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data = data.drop(columns=['PassengerId', 'Name', 'Age', 'Cabin', 'Ticket'])

In [ ]:
data.head()

In [ ]:
data['Embarked'].value_counts()

In [ ]:
data['Embarked'] =data['Embarked'].replace({'S':"Southampton", 'C':'Chebourg', 'Q':"Queenstown"})

In [ ]:
data.dropna(subset=['Embarked'] , inplace=True)

In [ ]:
data['Fare'] = data['Fare'].astype('int8')

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
label = LabelEncoder()

In [ ]:
data['Sex'] = label.fit_transform(data['Sex'])

In [ ]:
onehot = OneHotEncoder(sparse_output=False)

In [ ]:
Embarked = onehot.fit_transform(data[['Embarked']])

In [ ]:
Embarked = pd.DataFrame(Embarked, columns=onehot.get_feature_names_out())

In [ ]:
Embarked

In [ ]:
data = pd.concat([data.drop(columns=['Embarked']),Embarked], axis=1)

In [ ]:
data.sample(5)

In [ ]:
data.isnull().sum()

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scale = StandardScaler()

In [ ]:
data.columns

In [ ]:
num_cols = ['Pclass',  'SibSp', 'Parch', 'Fare']

In [ ]:
data[num_cols] = scale.fit_transform(data[num_cols])

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
data = data.dropna()

In [ ]:
data.isnull().sum()

In [ ]:
X = data.drop(columns=['Survived'])

In [ ]:
y = data['Survived']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test , y_train, y_test = train_test_split(X,y, test_size=20, random_state=42)

In [ ]:
X_train, X_valid, y_train, y_valid =  train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
import tensorflow

In [ ]:
(X_train.shape[1],)

In [ ]:
model = Sequential([Dense(128, input_shape=(X_train.shape[1],), activation='relu'), # First Hidden Layer
Dense(64, activation='relu'),
Dense(32, activation='relu'),
Dense(1, activation='sigmoid')
])

In [ ]:
model2 = Sequential([Dense(128, input_shape=(X_train.shape[1],), activation='relu'), # First Hidden Layer
Dense(64, activation='relu'),
Dense(32, activation='relu'),
Dense(1, activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
opt  = tensorflow.keras.optimizers.Adam(learning_rate=0.01)

In [ ]:
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
X_train.isnull().sum()

In [ ]:
model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=50)

In [ ]:
# model2.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=100)

In [ ]:
# SGDRMSprop
# Adam
# AdamW
# Adadelta
# Adagrad
# Adamax

In [ ]:
def build_model(hp):
  model = Sequential([
      Dense( 64, input_shape=(X_train.shape[1],), activation='relu'),
      Dense( 32, activation='relu'),
      Dense( 1, activation='sigmoid'),
  ])

  optimizer = hp.Choice('optimizers', values=['SGD','RMSprop', 'Adam', 'AdamW', 'Adadelta','Adagrad','Adamax'])

  model.compile(optimizer = optimizer, loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [ ]:
tuner = kt.RandomSearch(build_model, max_trials = 5, objective='val_accuracy')

In [ ]:
tuner.search(X_train, y_train, validation_data = (X_test, y_test), epochs = 10)

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.fit(X_train, y_train, validation_data = (X_test, y_test), epochs =100, initial_epoch=11)

Choose the Most Ideal Numbers of NOdes we should have in a hidden layer

In [ ]:
from tensorflow.keras.layers import Input
tensorflow.config.run_functions_eagerly(True)

In [ ]:
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)

In [ ]:
def build_model(hp):
  nodes = hp.Int('nodes',8,128,8
  )
  activation_func = hp.Choice("activation", values=['sigmoid', 'tanh'])

  model = Sequential()
  model.add(Input(shape=(X_train.shape[1],)))
  model.add(Dense(nodes, activation='relu'))
  model.add(Dense(nodes,  activation='relu'))
  model.add(Dense(1, activation= activation_func))

  model.compile(optimizer='rmsprop', metrics=['accuracy'], loss = 'binary_crossentropy')

  return model


In [ ]:
tuner = kt.RandomSearch(build_model, objective='val_loss', max_trials=5, directory = 'nodes1', project_name = 'nodes_details')

In [ ]:
tuner.search(X_train, y_train , validation_data=(X_test, y_test), epochs = 10)

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
model = tuner.get_best_models(num_models = 1)[0]

In [ ]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs = 100, initial_epoch=11)

## Number of Optimal Hidden Layers

In [ ]:
from tensorflow.keras.layers import Input, Dropout, Dense

In [ ]:
def build_model(hp):
  model= Sequential()
  model.add(Input(shape =(X_train.shape[1],)))

  for i in range(hp.Int('hidden', min_value = 1, max_value = 10, step = 1)):
    model.add(Dense(88, activation='relu'))

  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
  return model



In [ ]:
tunner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5, directory='hidden')

In [ ]:
tuner.search(X_train, y_train , validation_data=(X_test, y_test), epochs = 10)

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs = 100, initial_epoch=11)

## Hyperparameter Tuning
1. No. of Hidden layers
2. No. of nodes in Hidden layers
3. Optimal Optimizer
4. Dropout Layer - Value

In [ ]:
def build_model(hp):
  model = Sequential()
  optimizer = hp.Choice("optimizer", values=['SGD','RMSprop', 'Adam', 'AdamW', 'Adadelta','Adagrad','Adamax'])
  model.add(Input(shape=(X_train.shape[1],)))
  for i in range(hp.Int('hidden', min_value=1, max_value=10, step=1)):
    nodes = hp.Int('nodes', min_value=8, max_value = 128, step=8)
    model.add(Dense(nodes, activation='relu'))

    dropout_val = hp.Float('drop', min_value = 0.1, max_value = 0.9, step = 1)
    model.add(Dropout(dropout_val))

  model.add(Dense(1, activation='sigmoid'))


  model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

  return model


In [ ]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5, directory='all_in_one1')

In [ ]:
tuner.search(X_train, y_train , validation_data=(X_test, y_test), epochs = 10)

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs = 100, batch_size=32)